# Do Surrogate Models Recover the Truth?
## Evaluating Global Surrogate Explanations against WHO Growth Standard Rules in Child Stunting Classification

Companion notebook for the manuscript. In the Kaggle *Stunting Toddler (Balita) Detection (121K rows)* dataset, the label is a deterministic function of age, sex and height through the WHO length/height-for-age z-score (HAZ). The **true explanation is therefore known**. This notebook tests how well global surrogate decision trees, fitted to black-box classifiers, recover that rule.

**Sections**
1. Configuration and data files
2. WHO growth standard (LMS) tables
3. Data loading and cleaning (Table 1)
4. Ground-truth audit: recompute HAZ and check the labels (Table 2, Fig. 1)
5. Leakage-safe split and black-box models (Table 3)
6. Global surrogates: fidelity and complexity (Table 4, Fig. 2)
7. Ground-truth recovery: decision boundaries on a dense grid (Table 5, Figs. 3–4)
8. Feature space with HAZ: can the surrogate recover the exact thresholds? (Table 6)
9. Controlled simulation: label noise × sample size (Table 7, Fig. 5)
10. Bootstrap stability of recovered rules (Table 8, Fig. 6)
11. Export

**Data you need (both free):**
- Kaggle dataset: search *"stunting-balita-detection-121k-rows"* on kaggle.com, download the CSV, and save it as `data/data_balita.csv`.
- WHO length/height-for-age z-score tables (0–2 and 2–5 years, boys and girls). The notebook downloads them automatically from cdn.who.int. If the download fails, download them manually from https://www.who.int/toolkits/child-growth-standards/standards/length-height-for-age and put them in `who/`.

## 1. Configuration

In [ ]:
import os, json, time, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, recall_score)
warnings.filterwarnings("ignore")

DATA_FILE  = "data/data_balita.csv"   # Kaggle CSV
WHO_DIR    = "who"                    # WHO LMS tables (auto-downloaded)
OUT        = "outputs_paper6"
SEED       = 2026
SMOKE_TEST = False   # True = run on SYNTHETIC data with APPROXIMATE LMS (code check only, never report)
QUICK      = False   # True = fewer depths / reps / bootstraps (fast check)

DEPTHS        = list(range(1, 15))
FID_TOL       = 0.01        # chosen depth: smallest depth within 1 pp of best CV balanced fidelity
MIN_LEAF_FRAC = 0.001       # min samples per leaf (fraction of explanation set)
GRID_STEP_CM  = 0.1
GRID_Z_RANGE  = (-5.0, 5.0) # evaluation domain around the WHO median
BAND_Z        = 0.5         # "boundary band" half-width in z units
SIM_N         = [1000, 5000, 20000, 80000]
SIM_NOISE     = [0.0, 0.02, 0.05, 0.10]
SIM_REPS      = 3
N_BOOT        = 20
CLASS_NAMES   = ["severely stunted", "stunted", "normal", "tall"]

if QUICK:
    DEPTHS = [2, 4, 6, 8, 10, 12]; SIM_N = [1000, 10000]; SIM_NOISE = [0.0, 0.05]; SIM_REPS = 1; N_BOOT = 3
os.makedirs(OUT, exist_ok=True); os.makedirs(WHO_DIR, exist_ok=True); os.makedirs("data", exist_ok=True)
rng = np.random.default_rng(SEED)
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 110})

## 2. WHO growth standard (LMS) tables
The WHO length/height-for-age standard uses the LMS method. For a child of age *a* and sex *s*, with table values L, M, S:

$z = \dfrac{(x/M)^L - 1}{L\,S}$ (for LHFA, L = 1, so $z = (x/M - 1)/S$), and the height at a given z is $x(z) = M(1 + L S z)^{1/L}$.

Children under 24 months are measured lying down (length, 0–2 y table) and from 24 months standing (height, 2–5 y table). Both tables contain month 24, and section 4 checks which one the dataset used.

In [ ]:
WHO_BASE = "https://cdn.who.int/media/docs/default-source/child-growth/child-growth-standards/indicators/length-height-for-age/"
WHO_URLS = {  # full links as listed on the WHO page (including the ?sfvrsn= version parameter)
 ("M", "0-2"): WHO_BASE + "lhfa_boys_0-to-2-years_zscores.xlsx?sfvrsn=30e044c_9",
 ("M", "2-5"): WHO_BASE + "lhfa_boys_2-to-5-years_zscores.xlsx?sfvrsn=17e5ad91_9",
 ("F", "0-2"): WHO_BASE + "lhfa_girls_0-to-2-years_zscores.xlsx?sfvrsn=e9e66a95_11",
 ("F", "2-5"): WHO_BASE + "lhfa_girls_2-to-5-years_zscores.xlsx?sfvrsn=2ec187b9_11",
}
# Manual alternative: open https://www.who.int/toolkits/child-growth-standards/standards/length-height-for-age ,
# section "Tables > z-scores", click "Excel" for Boys/Girls "Birth to 2 years" and "2 to 5 years"
# (4 files) and put them in the who/ folder. Any file name is accepted if it contains
# boys/girls and 0-to-2 / 2-to-5 (e.g. lhfa_boys_0-to-2-years_zscores.xlsx).

import glob, re
def _find_local(sex, part):
    sx = "boys" if sex == "M" else "girls"
    pat = r"0[-_ ]?to[-_ ]?2" if part == "0-2" else r"2[-_ ]?to[-_ ]?5"
    for f in glob.glob(os.path.join(WHO_DIR, "*.xls*")):
        n = os.path.basename(f).lower()
        if sx in n and re.search(pat, n) and "percentile" not in n and "_p_" not in n:
            return f
    return None

def _download(url, path):
    req = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        "Accept": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,*/*",
        "Referer": "https://www.who.int/toolkits/child-growth-standards/standards/length-height-for-age"})
    with urllib.request.urlopen(req, timeout=60) as r, open(path, "wb") as fh:
        fh.write(r.read())

def _read_who_table(path):
    df = pd.read_excel(path)
    cols = {c.strip().lower(): c for c in df.columns.astype(str)}
    need = {k: cols[k] for k in ["month", "l", "m", "s"] if k in cols}
    if len(need) < 4:
        raise ValueError(f"{path}: expected columns Month, L, M, S; found {list(df.columns)}")
    t = df[[need["month"], need["l"], need["m"], need["s"]]].copy()
    t.columns = ["month", "L", "M", "S"]
    return t.dropna().astype(float)

def approx_lms():
    """APPROXIMATE LMS for SMOKE TESTS ONLY (interpolated anchor medians; L = 1)."""
    anchors = {"M": {0: 49.9, 3: 61.4, 6: 67.6, 9: 72.0, 12: 75.7, 18: 82.3, 24: 87.8, 36: 96.1, 48: 103.3, 60: 110.0},
               "F": {0: 49.1, 3: 59.8, 6: 65.7, 9: 70.1, 12: 74.0, 18: 80.7, 24: 86.4, 36: 95.1, 48: 102.7, 60: 109.4}}
    s_anchor = {0: .038, 3: .035, 12: .034, 24: .036, 36: .039, 60: .042}
    out = {}
    for sex, a in anchors.items():
        m = np.interp(np.arange(61), list(a), list(a.values()))
        s = np.interp(np.arange(61), list(s_anchor), list(s_anchor.values()))
        len_t = pd.DataFrame({"month": np.arange(0, 25), "L": 1.0, "M": m[:25], "S": s[:25]})
        ht = m[24:].copy(); ht[0] -= 0.7
        ht_t = pd.DataFrame({"month": np.arange(24, 61), "L": 1.0, "M": ht, "S": s[24:]})
        out[(sex, "0-2")], out[(sex, "2-5")] = len_t, ht_t
    return out

def load_who():
    if SMOKE_TEST:
        print("SMOKE_TEST: using APPROXIMATE LMS values - results are NOT valid for publication")
        return approx_lms()
    tabs, missing = {}, []
    for (sex, part), url in WHO_URLS.items():
        path = _find_local(sex, part)
        if path is None:
            path = os.path.join(WHO_DIR, os.path.basename(url.split("?")[0]))
            try:
                print("downloading", os.path.basename(path)); _download(url, path)
            except Exception as e:
                if os.path.exists(path): os.remove(path)
                missing.append((sex, part, url, str(e))); continue
        print("using", path)
        tabs[(sex, part)] = _read_who_table(path)
    if missing:
        msg = "\n".join(f"  - {'boys' if s=='M' else 'girls'} {p}: {u}" for s, p, u, _ in missing)
        raise SystemExit("Automatic download failed for:\n" + msg +
                         f"\nDownload these Excel files with your browser and save them in the folder '{os.path.abspath(WHO_DIR)}', "
                         "then run this cell again.")
    return tabs

WHO = load_who()

def lms_lookup(age_m, sex, month24="height"):
    """Vectorised L, M, S for integer months; month 24 from the length (0-2) or height (2-5) table."""
    age_m = np.asarray(age_m).astype(int); sex = np.asarray(sex)
    L = np.full(age_m.shape, np.nan); M = L.copy(); S = L.copy()
    for sx in ("M", "F"):
        t02 = WHO[(sx, "0-2")].set_index("month"); t25 = WHO[(sx, "2-5")].set_index("month")
        for a in np.unique(age_m[sex == sx]):
            use25 = a > 24 or (a == 24 and month24 == "height")
            row = (t25 if use25 else t02).loc[float(a)]
            idx = (age_m == a) & (sex == sx)
            L[idx], M[idx], S[idx] = row["L"], row["M"], row["S"]
    return L, M, S

def haz(height, age_m, sex, month24="height"):
    L, M, S = lms_lookup(age_m, sex, month24)
    x = np.asarray(height, float)
    return np.where(np.abs(L) > 1e-9, ((x / M) ** L - 1) / (L * S), np.log(x / M) / S)

def height_at_z(z, age_m, sex, month24="height"):
    L, M, S = lms_lookup(age_m, sex, month24)
    return np.where(np.abs(L) > 1e-9, M * (1 + L * S * z) ** (1 / L), M * np.exp(S * z))

print({k: v.shape for k, v in WHO.items()})

## 3. Data loading and cleaning (Table 1)

In [ ]:
def simulate_dataset(n, rng, noise=0.0, month24="height", tall_cut=3.0, meas_sd=0.0):
    """Synthetic children labelled by the WHO rule (used for SMOKE_TEST and section 9)."""
    age = rng.integers(0, 61, n); sex = np.where(rng.random(n) < .5, "M", "F")
    # z mixture roughly matching the Kaggle class mix (16% / 11% / 56% / 16%)
    comp = rng.choice(4, n, p=[.16, .11, .57, .16])
    z = np.select([comp == 0, comp == 1, comp == 2, comp == 3],
                  [rng.uniform(-5, -3, n), rng.uniform(-3, -2, n), rng.uniform(-2, 3, n), rng.uniform(3, 5, n)])
    h = np.round(height_at_z(z, age, sex, month24) + rng.normal(0, meas_sd, n), 1)
    y = rule_label(haz(h, age, sex, month24), tall_cut)
    if noise > 0:  # flip to an adjacent class
        f = rng.random(n) < noise
        step = np.where(rng.random(n) < .5, -1, 1)
        y = np.where(f, np.clip(y + step, 0, 3), y)
    return pd.DataFrame({"age_m": age, "sex": sex, "height_cm": h, "label": y})

def rule_label(z, tall_cut=3.0):
    """Permenkes 2/2020 TB/U: <-3 severely stunted; -3..<-2 stunted; -2..+3 normal; >+3 tall."""
    z = np.asarray(z)
    return np.select([z < -3, z < -2, z <= tall_cut], [0, 1, 2], default=3)

LABEL_MAP = {"severely stunted": 0, "sangat pendek": 0, "stunted": 1, "pendek": 1,
             "normal": 2, "tinggi": 3, "tall": 3}

def load_kaggle(path):
    df = pd.read_csv(path)
    pick = lambda keys: next(c for c in df.columns if any(k in c.lower() for k in keys))
    c_age, c_sex = pick(["umur", "age"]), pick(["kelamin", "sex", "gender"])
    c_h, c_y = pick(["tinggi", "height"]), pick(["status", "gizi", "label"])
    out = pd.DataFrame({
        "age_m": pd.to_numeric(df[c_age], errors="coerce"),
        "sex": df[c_sex].astype(str).str.lower().map(lambda s: "F" if s.startswith(("p", "f", "w")) else "M"),
        "height_cm": pd.to_numeric(df[c_h], errors="coerce"),
        "label": df[c_y].astype(str).str.strip().str.lower().map(LABEL_MAP)})
    print("columns used:", c_age, "|", c_sex, "|", c_h, "|", c_y)
    unk = df[c_y][out["label"].isna()].unique()
    if len(unk): print("[warn] unmapped labels:", unk)
    return out

if SMOKE_TEST:
    raw = simulate_dataset(121000 if not QUICK else 30000, rng, noise=0.0)
else:
    raw = load_kaggle(DATA_FILE)

n0 = len(raw)
df = raw.dropna().copy()
df = df[(df.age_m >= 0) & (df.age_m <= 60)]
df["age_m"] = df["age_m"].round().astype(int)
df["sex_f"] = (df["sex"] == "F").astype(int)
df["key"] = df["age_m"].astype(str) + "_" + df["sex"] + "_" + df["height_cm"].round(1).astype(str)
conflict = df.groupby("key")["label"].nunique()
T1 = pd.DataFrame({
    "Rows in file": [n0], "Rows after cleaning": [len(df)],
    "Exact duplicate rows": [int(df.duplicated(["age_m", "sex", "height_cm", "label"]).sum())],
    "Unique (age, sex, height) vectors": [df["key"].nunique()],
    "Vectors with conflicting labels": [int((conflict > 1).sum())],
    "Age range (months)": [f"{df.age_m.min()}–{df.age_m.max()}"],
    "Height range (cm)": [f"{df.height_cm.min():.1f}–{df.height_cm.max():.1f}"],
    "Female (%)": [round(100 * df.sex_f.mean(), 1)]}).T
T1.columns = ["Value"]
cls = df["label"].value_counts().sort_index()
for k, v in cls.items(): T1.loc[f"Class: {CLASS_NAMES[int(k)]}"] = f"{v} ({100*v/len(df):.1f}%)"
T1.to_csv(f"{OUT}/T1_dataset.csv"); T1

## 4. Ground-truth audit (Table 2, Fig. 1)
HAZ is recomputed from age, sex and height. The label is then reconstructed under alternative rule specifications (cut-off for *tall*, and which table is used at 24 months). The specification with the highest concordance becomes the **ground-truth rule** g\*.

In [ ]:
variants = []
for tall_cut in (3.0, 2.0):
    for m24 in ("height", "length"):
        z = haz(df.height_cm.values, df.age_m.values, df.sex.values, m24)
        yr = rule_label(z, tall_cut)
        variants.append(dict(rule=f"tall>{'+' if tall_cut>0 else ''}{tall_cut:g} SD, month24={m24}",
                             tall_cut=tall_cut, month24=m24,
                             concordance=accuracy_score(df.label, yr),
                             kappa=cohen_kappa_score(df.label, yr),
                             n_disagree=int((df.label.values != yr).sum())))
T2 = pd.DataFrame(variants).sort_values("concordance", ascending=False).reset_index(drop=True)
BEST = T2.iloc[0].to_dict()
TALL, M24 = BEST["tall_cut"], BEST["month24"]
df["haz"] = haz(df.height_cm.values, df.age_m.values, df.sex.values, M24)
df["y_rule"] = rule_label(df["haz"].values, TALL)
T2.to_csv(f"{OUT}/T2_label_audit.csv", index=False)
print("Ground-truth rule:", BEST["rule"], f"| concordance = {BEST['concordance']:.5f}")

cm = pd.DataFrame(confusion_matrix(df.label, df.y_rule, labels=[0, 1, 2, 3]),
                  index=[f"given: {c}" for c in CLASS_NAMES], columns=[f"rule: {c}" for c in CLASS_NAMES])
cm.to_csv(f"{OUT}/T2b_confusion_given_vs_rule.csv")
dis = df[df.label != df.y_rule]
if len(dis):
    cuts = np.array([-3, -2, TALL])
    dis_dist = np.min(np.abs(dis["haz"].values[:, None] - cuts[None, :]), axis=1)
    print(f"Disagreements: {len(dis)}; median |z - nearest cut-off| = {np.median(dis_dist):.3f}")
    print(dis.groupby("age_m").size().sort_values(ascending=False).head(10))
print(T2.to_string())
cm

In [ ]:
# Fig. 1: data with the true WHO boundaries
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
cols = ["#B2182B", "#EF8A62", "#67A9CF", "#2166AC"]
ages = np.arange(0, 61)
sub = df.sample(min(20000, len(df)), random_state=SEED)
for i, sx in enumerate(["M", "F"]):
    s = sub[sub.sex == sx]
    for k in range(4):
        q = s[s.label == k]; ax[i].scatter(q.age_m + rng.uniform(-.3, .3, len(q)), q.height_cm, s=1, c=cols[k], label=CLASS_NAMES[k])
    for zc in (-3, -2, TALL):
        ax[i].plot(ages, height_at_z(zc, ages, np.repeat(sx, 61), M24), "k-", lw=.8)
    ax[i].set(title=["Boys", "Girls"][i], xlabel="Age (months)")
ax[0].set_ylabel("Length/height (cm)"); ax[0].legend(markerscale=6, fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/F1_data_and_true_boundaries.png", dpi=220); plt.show()

## 5. Leakage-safe split and black-box models (Table 3)
Rows with the same (age, sex, height) vector are kept in the same split (grouped split), so that duplicates cannot appear in both training and test. The training part is divided into a **fit set** (black-box training) and an **explanation set** (surrogate training only). For comparison, a naive random split is also reported.

In [ ]:
X_COLS = ["age_m", "sex_f", "height_cm"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, te_idx = next(gss.split(df, groups=df["key"]))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
f_idx, e_idx = next(gss2.split(train, groups=train["key"]))
fit, expl = train.iloc[f_idx].reset_index(drop=True), train.iloc[e_idx].reset_index(drop=True)
print(f"fit={len(fit)}, explanation={len(expl)}, test={len(test)}; "
      f"shared keys fit/test = {len(set(fit.key) & set(test.key))}")

def make_models(seed=SEED):
    m = {"RF": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=seed),
         "GBM": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=seed),
         "MLP": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=300,
                                                              early_stopping=True, random_state=seed))}
    try:
        from xgboost import XGBClassifier
        m["XGB"] = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, random_state=seed, n_jobs=-1)
    except ImportError:
        pass
    return m

def perf(y, yh):
    return dict(Accuracy=accuracy_score(y, yh), BalancedAcc=balanced_accuracy_score(y, yh),
                MacroF1=f1_score(y, yh, average="macro"),
                **{f"Recall_{CLASS_NAMES[k]}": recall_score(y, yh, labels=[k], average="macro", zero_division=0) for k in range(4)})

BB, rows = {}, []
for name, mdl in make_models().items():
    t0 = time.time(); mdl.fit(fit[X_COLS], fit["label"]); t = time.time() - t0
    BB[name] = mdl
    yh = mdl.predict(test[X_COLS])
    rows.append(dict(Model=name, Split="grouped", **perf(test["label"], yh),
                     AgreeWithRule=accuracy_score(test["y_rule"], yh), FitTime_s=round(t, 1)))
# naive random split (duplicates may leak) - RF only, for contrast
a, b = train_test_split(df, test_size=.3, random_state=SEED, stratify=df["label"])
rf_naive = clone(make_models()["RF"]).fit(a[X_COLS], a["label"])
rows.append(dict(Model="RF", Split="naive random", **perf(b["label"], rf_naive.predict(b[X_COLS])),
                 AgreeWithRule=accuracy_score(b["y_rule"], rf_naive.predict(b[X_COLS])), FitTime_s=np.nan))
T3 = pd.DataFrame(rows); T3.to_csv(f"{OUT}/T3_blackbox_performance.csv", index=False); T3.round(4)

## 6. Global surrogates: fidelity and complexity (Table 4, Fig. 2)
A CART tree *g* is fitted on the explanation set to the black box's predicted labels. For each depth we report fidelity to the black box and agreement with the true rule on the test set. An **oracle tree**, fitted directly to the true-rule labels at the same depth, gives the best that a tree of that depth can do.

In [ ]:
def tree(d, n):
    return DecisionTreeClassifier(max_depth=d, min_samples_leaf=max(1, int(MIN_LEAF_FRAC * n)), random_state=SEED)

def choose_depth(X, yb):
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED); res = []
    for d in DEPTHS:
        f = [balanced_accuracy_score(yb[v], tree(d, len(t)).fit(X.iloc[t], yb[t]).predict(X.iloc[v]))
             for t, v in cv.split(X, yb)]
        res.append((d, np.mean(f)))
    best = max(r[1] for r in res)
    return min(d for d, f in res if f >= best - FID_TOL)

SUR, ORACLE, DSTAR, sweep = {}, {}, {}, []
Xe, Xt = expl[X_COLS], test[X_COLS]
for name, mdl in BB.items():
    yb_e, yb_t = mdl.predict(Xe), mdl.predict(Xt)
    for d in DEPTHS:
        g = tree(d, len(Xe)).fit(Xe, yb_e); o = tree(d, len(Xe)).fit(Xe, expl["y_rule"])
        gt = g.predict(Xt)
        sweep.append(dict(Model=name, depth=d, leaves=g.get_n_leaves(),
                          Fidelity=accuracy_score(yb_t, gt), BalFidelity=balanced_accuracy_score(yb_t, gt),
                          TruthAgree_surrogate=accuracy_score(test["y_rule"], gt),
                          TruthAgree_oracle=accuracy_score(test["y_rule"], o.predict(Xt))))
    DSTAR[name] = choose_depth(Xe, yb_e)
    SUR[name] = tree(DSTAR[name], len(Xe)).fit(Xe, yb_e)
    ORACLE[name] = tree(DSTAR[name], len(Xe)).fit(Xe, expl["y_rule"])
    with open(f"{OUT}/rules_{name}_depth{DSTAR[name]}.txt", "w") as fh:
        fh.write(export_text(SUR[name], feature_names=X_COLS, show_weights=True))
SW = pd.DataFrame(sweep); SW.to_csv(f"{OUT}/T4b_depth_sweep.csv", index=False)
T4 = SW.merge(pd.DataFrame({"Model": list(DSTAR), "depth": list(DSTAR.values())}), on=["Model", "depth"])
T4["n_features_used"] = [len(set(SUR[m].tree_.feature[SUR[m].tree_.feature >= 0])) for m in T4.Model]
T4["uses_sex"] = [bool((SUR[m].tree_.feature == 1).any()) for m in T4.Model]
T4.to_csv(f"{OUT}/T4_surrogate_fidelity.csv", index=False); T4.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for name in BB:
    s = SW[SW.Model == name]
    ax[0].plot(s.depth, s.BalFidelity, "o-", ms=3, label=name)
    ax[1].plot(s.depth, s.TruthAgree_surrogate, "o-", ms=3, label=name)
s = SW[SW.Model == list(BB)[0]]
ax[1].plot(s.depth, s.TruthAgree_oracle, "k--", label="Oracle tree (true labels)")
ax[0].set(xlabel="Surrogate depth", ylabel="Balanced fidelity to black box (test)")
ax[1].set(xlabel="Surrogate depth", ylabel="Agreement with WHO rule (test)")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/F2_fidelity_truth_vs_depth.png", dpi=220); plt.show()

## 7. Ground-truth recovery on a dense grid (Table 5, Figs. 3–4)
For every age (0–60 months) and sex, heights are scanned from z = −5 to +5 in 0.1 cm steps. For each explainer we locate the recovered boundaries ĥ_k (first height at which the predicted class reaches class k) and compare them with the true WHO cut-off heights h\*_k:
- **MAE (cm)** and **MAE (z)** of each boundary (severe/stunted at −3 SD, stunted/normal at −2 SD, normal/tall at +3 SD);
- **grid agreement** with the rule over the whole domain and within the *boundary band* |z − cut-off| < 0.5;
- **monotonicity violations**: share of (age, sex) columns where the predicted class is not non-decreasing in height;
- **sex effect recovery**: true vs recovered girl–boy difference in boundary heights.

In [ ]:
def make_grid():
    rows = []
    for sx in ("M", "F"):
        for a in range(61):
            lo, hi = height_at_z(np.array([GRID_Z_RANGE[0], GRID_Z_RANGE[1]]), np.array([a, a]), np.array([sx, sx]), M24)
            h = np.round(np.arange(np.floor(lo * 10) / 10, hi + GRID_STEP_CM, GRID_STEP_CM), 1)
            rows.append(pd.DataFrame({"age_m": a, "sex": sx, "sex_f": int(sx == "F"), "height_cm": h}))
    g = pd.concat(rows, ignore_index=True)
    g["haz"] = haz(g.height_cm.values, g.age_m.values, g.sex.values, M24)
    g["y_rule"] = rule_label(g["haz"].values, TALL)
    return g
GRID = make_grid(); print("grid points:", len(GRID))
CUTS = {1: -3.0, 2: -2.0, 3: TALL}   # boundary k separates class k-1 from class k

def boundaries(pred):
    """Recovered boundary heights per (age, sex, k) from grid predictions."""
    G = GRID[["age_m", "sex", "height_cm"]].copy(); G["p"] = pred; out = []
    for (a, sx), s in G.groupby(["age_m", "sex"], sort=False):
        p = s["p"].values; h = s["height_cm"].values
        mono = bool(np.all(np.diff(p) >= 0))
        for k in (1, 2, 3):
            idx = np.argmax(p >= k) if (p >= k).any() else None
            out.append(dict(age_m=a, sex=sx, k=k, h_hat=h[idx] if idx is not None else np.nan, monotone=mono))
    B = pd.DataFrame(out)
    B["h_true"] = [float(height_at_z(CUTS[k], np.array([a]), np.array([sx]), M24)[0]) for a, sx, k in zip(B.age_m, B.sex, B.k)]
    B["z_hat"] = haz(B.h_hat.fillna(0).values, B.age_m.values, B.sex.values, M24)
    B.loc[B.h_hat.isna(), "z_hat"] = np.nan
    B["err_cm"] = B.h_hat - B.h_true; B["err_z"] = B.z_hat - B.k.map(CUTS)
    return B

def recovery_metrics(name, kind, pred):
    B = boundaries(pred)
    band = np.min(np.abs(GRID["haz"].values[:, None] - np.array(list(CUTS.values()))[None, :]), axis=1) < BAND_Z
    r = dict(Explainer=name, Kind=kind,
             GridAgree=accuracy_score(GRID.y_rule, pred), BandAgree=accuracy_score(GRID.y_rule[band], pred[band]),
             MonotoneViolation=1 - B.drop_duplicates(["age_m", "sex"]).monotone.mean())
    for k, zc in CUTS.items():
        q = B[B.k == k]
        r[f"MAE_cm_{zc:+g}"] = q.err_cm.abs().mean(); r[f"MAE_z_{zc:+g}"] = q.err_z.abs().mean()
        r[f"Bias_z_{zc:+g}"] = q.err_z.mean(); r[f"Missing_{zc:+g}"] = q.h_hat.isna().mean()
    piv = B.pivot_table(index=["age_m", "k"], columns="sex", values=["h_hat", "h_true"])
    r["SexDiff_true_cm"] = (piv[("h_true", "M")] - piv[("h_true", "F")]).abs().mean()
    r["SexDiff_recovered_cm"] = (piv[("h_hat", "M")] - piv[("h_hat", "F")]).abs().mean()
    return r, B

XG = GRID[X_COLS]
REC, BND = [], {}
for name, mdl in BB.items():
    for kind, est in [("black box", mdl), ("surrogate", SUR[name]), ("oracle tree", ORACLE[name])]:
        if kind == "oracle tree" and name != list(BB)[0]:
            continue  # the oracle does not depend on the black box (same depth reported once per depth)
        r, B = recovery_metrics(name, kind, est.predict(XG)); r["depth"] = DSTAR[name] if kind != "black box" else np.nan
        REC.append(r); BND[(name, kind)] = B
T5 = pd.DataFrame(REC); T5.to_csv(f"{OUT}/T5_truth_recovery.csv", index=False); T5.round(4)

In [ ]:
# Fig. 3: recovered vs true stunting boundary (-2 SD) for boys and girls
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
show = [("RF", "black box"), ("RF", "surrogate"), (list(BB)[-1], "surrogate")]
for i, sx in enumerate(["M", "F"]):
    B0 = BND[show[0]]; q = B0[(B0.sex == sx) & (B0.k == 2)]
    ax[i].plot(q.age_m, q.h_true, "k-", lw=2, label="WHO −2 SD (truth)")
    for (m, kind), st in zip(show, ["-", "--", ":"]):
        q = BND[(m, kind)]; q = q[(q.sex == sx) & (q.k == 2)]
        ax[i].step(q.age_m, q.h_hat, st, where="mid", label=f"{m} {kind}")
    ax[i].set(title=["Boys", "Girls"][i], xlabel="Age (months)")
ax[0].set_ylabel("Height at stunting boundary (cm)"); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/F3_boundary_recovery.png", dpi=220); plt.show()

# Fig. 4: boundary error in z units by age (surrogates)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for j, (k, zc) in enumerate(CUTS.items()):
    for name in BB:
        q = BND[(name, "surrogate")]; q = q[q.k == k].groupby("age_m").err_z.apply(lambda v: v.abs().mean())
        ax[j].plot(q.index, q.values, label=name)
    ax[j].set(title=f"Boundary at {zc:+g} SD", xlabel="Age (months)")
ax[0].set_ylabel("|error| (z units)"); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/F4_boundary_error_by_age.png", dpi=220); plt.show()

## 8. Feature space with HAZ (Table 6)
When HAZ is added as a feature, the true rule needs only three splits on HAZ (−3, −2, +3) and four leaves. Here we test whether surrogates find this parsimonious rule and how close their thresholds are.

In [ ]:
XH = ["age_m", "sex_f", "height_cm", "haz"]
rows = []
for name, proto in make_models().items():
    m = clone(proto).fit(fit[XH], fit["label"])
    yb_e, yb_t = m.predict(expl[XH]), m.predict(test[XH])
    for d in (2, 3, 4, 6):
        g = tree(d, len(expl)).fit(expl[XH], yb_e); t = g.tree_
        thr = np.sort(t.threshold[t.feature == 3])
        err = [np.min(np.abs(thr - c)) if len(thr) else np.nan for c in CUTS.values()]
        rows.append(dict(Model=name, depth=d, leaves=g.get_n_leaves(),
                         Fidelity=accuracy_score(yb_t, g.predict(test[XH])),
                         TruthAgree=accuracy_score(test["y_rule"], g.predict(test[XH])),
                         HAZ_importance=g.feature_importances_[3],
                         HAZ_thresholds=str(np.round(thr, 3).tolist()),
                         ThrErr_m3=err[0], ThrErr_m2=err[1], ThrErr_tall=err[2],
                         uses_only_HAZ=bool(set(t.feature[t.feature >= 0]) <= {3})))
T6 = pd.DataFrame(rows); T6.to_csv(f"{OUT}/T6_haz_feature_space.csv", index=False); T6.round(4)

## 9. Controlled simulation: label noise × sample size (Table 7, Fig. 5)
Synthetic children are generated from the same WHO tables and labelled by the same rule, with a share of labels flipped to an adjacent class. This shows how recovery degrades when the data are smaller or noisier than the Kaggle set, as in real posyandu records.

In [ ]:
sim_rows = []
srng = np.random.default_rng(SEED + 1)
test_sim = simulate_dataset(20000, srng, 0.0, M24, TALL)
for n in SIM_N:
    for eps in SIM_NOISE:
        for rep in range(SIM_REPS):
            d = simulate_dataset(n, srng, eps, M24, TALL)
            f_, e_ = train_test_split(d, test_size=.25, random_state=rep)
            for name in ["RF", "GBM"]:
                ff = f_.assign(sex_f=(f_.sex == "F").astype(int))
                m = clone(make_models(SEED + rep)[name]).fit(ff[X_COLS], ff["label"])
                ee = e_.assign(sex_f=(e_.sex == "F").astype(int))
                yb = m.predict(ee[X_COLS]); dd = choose_depth(ee[X_COLS].reset_index(drop=True), yb) if not QUICK else 8
                g = tree(dd, len(ee)).fit(ee[X_COLS], yb)
                r, B = recovery_metrics(name, "surrogate", g.predict(XG))
                ts = test_sim.assign(sex_f=(test_sim.sex == "F").astype(int))
                sim_rows.append(dict(n=n, noise=eps, rep=rep, Model=name, depth=dd,
                                     BB_TruthAgree=accuracy_score(ts.label, m.predict(ts[X_COLS])),
                                     SUR_TruthAgree=r["GridAgree"], SUR_BandAgree=r["BandAgree"],
                                     MAE_z_stunting=r["MAE_z_-2"], MAE_cm_stunting=r["MAE_cm_-2"],
                                     MonotoneViolation=r["MonotoneViolation"]))
SIM = pd.DataFrame(sim_rows)
T7 = SIM.groupby(["Model", "n", "noise"]).agg(["mean", "std"]).drop(columns=["rep"], level=0)
T7.to_csv(f"{OUT}/T7_simulation.csv"); SIM.to_csv(f"{OUT}/T7_simulation_raw.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for name, st in zip(["RF", "GBM"], ["-", "--"]):
    for eps in SIM_NOISE:
        q = SIM[(SIM.Model == name) & (SIM.noise == eps)].groupby("n").MAE_z_stunting.mean()
        ax[0].plot(q.index, q.values, st, marker="o", ms=3, label=f"{name}, noise {eps:.0%}")
        q = SIM[(SIM.Model == name) & (SIM.noise == eps)].groupby("n").SUR_BandAgree.mean()
        ax[1].plot(q.index, q.values, st, marker="o", ms=3)
for a_ in ax: a_.set_xscale("log"); a_.set_xlabel("Training size n")
ax[0].set_ylabel("MAE of −2 SD boundary (z)"); ax[1].set_ylabel("Boundary-band agreement with rule")
ax[0].legend(fontsize=6); plt.tight_layout(); plt.savefig(f"{OUT}/F5_simulation.png", dpi=220); plt.show()
T7.round(4)

## 10. Bootstrap stability of recovered rules (Table 8, Fig. 6)
The black box (RF) is refitted on bootstrap resamples of the fit set and the surrogate on resamples of the explanation set. We record the root split, the number of leaves and the boundary errors.

In [ ]:
brows = []
for b in range(N_BOOT):
    fb = fit.sample(len(fit), replace=True, random_state=b); eb = expl.sample(len(expl), replace=True, random_state=1000 + b)
    m = clone(make_models(SEED + b)["RF"]).fit(fb[X_COLS], fb["label"])
    g = tree(DSTAR["RF"], len(eb)).fit(eb[X_COLS], m.predict(eb[X_COLS]))
    r, _ = recovery_metrics("RF", "surrogate", g.predict(XG)); t = g.tree_
    brows.append(dict(boot=b, root_feature=X_COLS[t.feature[0]], root_threshold=t.threshold[0], leaves=g.get_n_leaves(),
                      GridAgree=r["GridAgree"], MAE_z_m3=r["MAE_z_-3"], MAE_z_m2=r["MAE_z_-2"], MAE_z_tall=r[f"MAE_z_{TALL:+g}"]))
BOOT = pd.DataFrame(brows); BOOT.to_csv(f"{OUT}/T8_bootstrap_raw.csv", index=False)
T8 = pd.DataFrame({"Root feature (mode)": [BOOT.root_feature.mode()[0]],
                   "Root agreement": [BOOT.root_feature.eq(BOOT.root_feature.mode()[0]).mean()],
                   "Leaves mean ± SD": [f"{BOOT.leaves.mean():.1f} ± {BOOT.leaves.std():.1f}"],
                   **{f"{c} mean [2.5–97.5%]": [f"{BOOT[c].mean():.3f} [{BOOT[c].quantile(.025):.3f}–{BOOT[c].quantile(.975):.3f}]"]
                      for c in ["GridAgree", "MAE_z_m3", "MAE_z_m2", "MAE_z_tall"]}}).T
T8.columns = ["RF surrogate"]; T8.to_csv(f"{OUT}/T8_bootstrap_stability.csv")
plt.figure(figsize=(5, 3.2)); plt.boxplot([BOOT.MAE_z_m3, BOOT.MAE_z_m2, BOOT.MAE_z_tall], tick_labels=["−3 SD", "−2 SD", f"+{TALL:g} SD"])
plt.ylabel("Boundary MAE (z)"); plt.title("Bootstrap variability of recovered boundaries (RF)")
plt.tight_layout(); plt.savefig(f"{OUT}/F6_bootstrap_stability.png", dpi=220); plt.show()
T8

## 11. Export
All tables (T1–T8) and figures (F1–F6) are in the output folder and map directly to the manuscript.

In [ ]:
fig = plt.figure(figsize=(16, 7))
plot_tree(SUR["RF"], feature_names=X_COLS, class_names=CLASS_NAMES, filled=True, impurity=False,
          proportion=True, max_depth=3, fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/F7_surrogate_tree_RF_top.png", dpi=220); plt.show()
summary = dict(ground_truth_rule=BEST["rule"], concordance=BEST["concordance"], depths=DSTAR, smoke_test=SMOKE_TEST)
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str)); print(sorted(os.listdir(OUT)))